In [1]:
# Cell 1: load everything
import pandas as pd
RAW = "../data/raw"
customers = pd.read_csv(f"{RAW}/olist_customers_dataset.csv")
orders    = pd.read_csv(f"{RAW}/olist_orders_dataset.csv")
items     = pd.read_csv(f"{RAW}/olist_order_items_dataset.csv")
payments  = pd.read_csv(f"{RAW}/olist_order_payments_dataset.csv")
reviews   = pd.read_csv(f"{RAW}/olist_order_reviews_dataset.csv")
products  = pd.read_csv(f"{RAW}/olist_products_dataset.csv")
sellers   = pd.read_csv(f"{RAW}/olist_sellers_dataset.csv")
geo       = pd.read_csv(f"{RAW}/olist_geolocation_dataset.csv")
cats      = pd.read_csv(f"{RAW}/product_category_name_translation.csv")

for name, df in [("customers", customers), ("orders", orders), ("items", items),
                 ("payments", payments), ("reviews", reviews), ("products", products),
                 ("sellers", sellers), ("geo", geo), ("cats", cats)]:
    print(f"{name:10s} {df.shape}")

customers  (99441, 5)
orders     (99441, 8)
items      (112650, 7)
payments   (103886, 5)
reviews    (99224, 7)
products   (32951, 9)
sellers    (3095, 4)
geo        (1000163, 5)
cats       (71, 2)


In [2]:
# Cell 2: customer_id vs customer_unique_id
print(customers["customer_id"].nunique(), customers["customer_unique_id"].nunique())

99441 96096


In [3]:
# Cell 3: duplicate review ids?
print(reviews["review_id"].duplicated().sum())
print(reviews.duplicated(subset=["review_id", "order_id"]).sum())

814
0


In [4]:
# Cell 4: typo'd column names
print(products.columns.tolist())

['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


In [5]:
# Cell 5: categories with no English translation
missing = set(products["product_category_name"].dropna()) - set(cats["product_category_name"])
print(missing)

{'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}


In [6]:
# Cell 6: order statuses and date range
print(orders["order_status"].value_counts())
print(orders["order_purchase_timestamp"].min(), orders["order_purchase_timestamp"].max())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
2016-09-04 21:15:19 2018-10-17 17:30:18


In [7]:
# Cell 7: do items and payments agree on money?
item_total = items.groupby("order_id")[["price", "freight_value"]].sum().sum(axis=1)
pay_total = payments.groupby("order_id")["payment_value"].sum()
diff = (item_total - pay_total).abs()
print((diff > 1).sum(), "orders where items and payments differ by > 1 BRL")

249 orders where items and payments differ by > 1 BRL


In [8]:
# Cell 8: geolocation rows per zip
print(geo.groupby("geolocation_zip_code_prefix").size().describe())

count    19015.000000
mean        52.598633
std         72.057907
min          1.000000
25%         10.000000
50%         29.000000
75%         66.500000
max       1146.000000
dtype: float64


In [9]:
# Cell 9: zip codes lose leading zeros if read as numbers
print(customers["customer_zip_code_prefix"].head())

0    14409
1     9790
2     1151
3     8775
4    13056
Name: customer_zip_code_prefix, dtype: int64
